# ARC Cluster Inspector

Browse any cluster — from the old scene-based clustering, the new process-based
clustering, or a manual task list.

**Usage:** set the config in Cell 2, then Run All.

- `MODE = 'cluster_id'` — show all tasks in a numbered cluster
- `MODE = 'task_list'`  — show a manual list of task IDs

- `CLUSTER_SOURCE = 'old'` — use `results/cluster_inspection.txt` (scene descriptions)
- `CLUSTER_SOURCE = 'new'` — use `results/cluster_process_inspection.txt` (process descriptions)


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────

MODE           = 'task_list'    # 'cluster_id' | 'task_list'
CLUSTER_SOURCE = 'old'          # 'old' | 'new'  (only used when MODE='cluster_id')
CLUSTER_ID     = 8              # cluster number to show (only used when MODE='cluster_id')

# Used when MODE='task_list' — paste any task IDs here, or use a preset below
TASK_IDS = [
    '09629e4f', '1190e5a7', '1e32b0e9', '272f95fa',
    '29623171', '54d9e175', '6773b310', '6d0160f0', '941d9a10',
]

# ── Presets (copy into TASK_IDS as needed) ────────────────────────────────────
_PRESETS = {
    'C8_curated':          ['09629e4f','1190e5a7','1e32b0e9','272f95fa','29623171','54d9e175','6773b310','6d0160f0','941d9a10'],
    'C8_outliers':         ['1bfc4729','7b6016b9'],
    'C3_mirror':           ['3af2c5a8','49d1d64f','4c4377d9','62c24649','67e8384a','6d0aefbc','6fa7a44f','7fe24cdd','8be77c9e','8d5021e8','a416b8f3','c9e6f938'],
    'pattern_restoration': ['3345333e','3631a71a','9ecd008a','b8825c91','0dfd9992','29ec7d0e','484b58aa','73251a56','c3f564a4'],
}

# ── Resolve task list ─────────────────────────────────────────────────────────
%matplotlib inline
import json, re
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

ROOT = Path('..').resolve()

ARC_COLORS = ['#000000','#0074D9','#FF4136','#2ECC40','#FFDC00',
               '#AAAAAA','#F012BE','#FF851B','#7FDBFF','#870C25']
CMAP = mcolors.ListedColormap(ARC_COLORS)
NORM = mcolors.BoundaryNorm(range(11), CMAP.N)
DATA_DIR = ROOT / 'data' / 'training'

def parse_cluster_file(path):
    """Return {cluster_id: [task_ids]} from a cluster inspection txt."""
    result = {}
    current = None
    for line in path.read_text().splitlines():
        m = re.match(r'^Cluster (\d+)', line)
        if m:
            current = int(m.group(1))
            result[current] = []
        m2 = re.match(r'^  ([0-9a-f]{8}):', line)
        if m2 and current is not None:
            result[current].append(m2.group(1))
    return result

if MODE == 'cluster_id':
    fname = 'cluster_process_inspection.txt' if CLUSTER_SOURCE == 'new' else 'cluster_inspection.txt'
    cfile = ROOT / 'results' / fname
    if not cfile.exists():
        raise FileNotFoundError(f'{cfile} not found — run compare_clusters.py first' if CLUSTER_SOURCE == 'new' else f'{cfile} not found')
    all_clusters = parse_cluster_file(cfile)
    task_ids = all_clusters.get(CLUSTER_ID, [])
    title = f'Cluster {CLUSTER_ID} ({CLUSTER_SOURCE}) — {len(task_ids)} tasks'
elif MODE == 'task_list':
    task_ids = TASK_IDS
    title = f'Task list — {len(task_ids)} tasks'
else:
    raise ValueError(f'Unknown MODE: {MODE!r}')

# Load descriptions for annotation
desc_file = ROOT / ('data/descriptions_process.json' if CLUSTER_SOURCE == 'new' else 'data/descriptions_scene.json')
descriptions = json.loads(desc_file.read_text()) if desc_file.exists() else {}

print(f'{title}')
print(f'Tasks: {task_ids}')


In [ ]:
# ── Show each task ────────────────────────────────────────────────────────────

def show_grid(ax, grid, title):
    g = np.array(grid)
    H, W = g.shape
    ax.imshow(g, cmap=CMAP, norm=NORM, interpolation='nearest')
    ax.set_title(title, fontsize=8)
    ax.set_xticks(np.arange(-0.5, W, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, H, 1), minor=True)
    ax.tick_params(which='minor', length=0)
    ax.grid(which='minor', color='white', linewidth=0.5)
    ax.set_xticks([]); ax.set_yticks([])

for tid in task_ids:
    task_path = DATA_DIR / f'{tid}.json'
    if not task_path.exists():
        print(f'!! {tid}: not found')
        continue
    task  = json.loads(task_path.read_text())
    pairs = task['train']
    n_rows = len(pairs) + 1

    fig, axes = plt.subplots(n_rows, 2, figsize=(6, 2.5 * n_rows), squeeze=False)
    fig.suptitle(tid, fontsize=9, y=1.01)

    for i, pair in enumerate(pairs):
        show_grid(axes[i, 0], pair['input'],  f'Train {i+1} in  {len(pair["input"])}×{len(pair["input"][0])}')
        show_grid(axes[i, 1], pair['output'], f'Train {i+1} out {len(pair["output"])}×{len(pair["output"][0])}')

    test = task['test'][0]
    show_grid(axes[-1, 0], test['input'], f'Test in  {len(test["input"])}×{len(test["input"][0])}')
    if 'output' in test:
        show_grid(axes[-1, 1], test['output'], 'Test out')
    else:
        axes[-1, 1].axis('off')
        axes[-1, 1].set_title('Test out (hidden)', fontsize=8)

    plt.tight_layout()
    plt.show()

    # Print description
    desc = descriptions.get(tid, '')
    if desc:
        for line in desc.strip().splitlines():
            print(f'  {line}')
    print()
